In [ ]:
# ── Cell A0: Load the combined benchmark ──────────────────────────────────────
import pandas as pd
from pathlib import Path

df = pd.read_csv("/kaggle/input/datasets/deboparna9das/csv-processing/human_queries_FINAL_combined.csv")
print(f"Loaded {len(df)} total queries")
print(df.columns.tolist())

In [ ]:
# ── Cell A1: Exact duplicate query text (all instances) ──────────────────────
dupe_groups = df[df.duplicated(subset="query", keep=False)].sort_values("query")
dupe_groups.to_csv("review_A_exact_duplicates.csv", index=False)
print(f"Exact duplicate query strings: {dupe_groups['query'].nunique()} unique texts, {len(dupe_groups)} total rows")
print(dupe_groups[["query_id","query","tool_name","image_id"]].to_string(index=False))

In [ ]:
# ── Cell A2: Flag duplicates with CONFLICTING labels (the dangerous cases) ───
conflict_check = dupe_groups.groupby("query")["tool_name"].nunique()
conflicting_queries = conflict_check[conflict_check > 1].index.tolist()
conflicts = dupe_groups[dupe_groups["query"].isin(conflicting_queries)]
conflicts.to_csv("review_A2_label_conflicts.csv", index=False)

print(f"⚠️ CONFLICTING duplicates (same exact text, different label): {len(conflicting_queries)} query texts, {len(conflicts)} rows")
if len(conflicts):
    print(conflicts[["query_id","query","tool_name","image_id"]].sort_values("query").to_string(index=False))
else:
    print("None found — clean on exact-match conflicts.")

In [ ]:
import pandas as pd

csv_path = "/kaggle/input/datasets/deboparna9das/parta1/review_A2_label_conflicts_RESOLVED.csv.csv"

df = pd.read_csv(csv_path)

dupe_groups = (
    df[df.duplicated(subset="query", keep=False)]
    .sort_values("query")
)

dupe_groups.to_csv("review_A_exact_duplicates.csv", index=False)

print(
    f"Exact duplicate query strings: "
    f"{dupe_groups['query'].nunique()} unique texts, "
    f"{len(dupe_groups)} total rows"
)

print(
    dupe_groups[
        ["query_id", "query", "tool_name", "image_id"]
    ].to_string(index=False)
)

In [ ]:
# ── Cell A3 (optional): Near-duplicate check — same normalized text, different casing/punctuation ─
import re

def normalize(q):
    return re.sub(r"[^\w\s]", "", str(q).lower()).strip()

df["_norm_query"] = df["query"].apply(normalize)
near_dupes = df[df.duplicated(subset="_norm_query", keep=False)].sort_values("_norm_query")
near_dupe_conflicts = near_dupes.groupby("_norm_query")["tool_name"].nunique()
near_conflicting = near_dupe_conflicts[near_dupe_conflicts > 1].index.tolist()
near_result = near_dupes[near_dupes["_norm_query"].isin(near_conflicting)]
near_result.to_csv("review_A3_near_duplicate_conflicts.csv", index=False)

print(f"Near-duplicate (normalized) conflicts: {len(near_conflicting)} groups, {len(near_result)} rows")
if len(near_result):
    print(near_result[["query_id","query","tool_name","image_id"]].sort_values("_norm_query").to_string(index=False))
df.drop(columns=["_norm_query"], inplace=True)

In [ ]:
# ── Cell A4: Apply your manual conflict resolutions ──────────────────────────
# Edit review_A2_label_conflicts.csv yourself first: for rows you want DROPPED,
# add a column "drop" = True. For rows where you changed the label, edit
# "tool_name" directly in that file.

resolved_A2 = pd.read_csv("/kaggle/input/datasets/deboparna9das/parta1/review_A2_label_conflicts_RESOLVED.csv.csv")

if "drop" in resolved_A2.columns:
    ids_to_drop = resolved_A2[resolved_A2["drop"] == True]["query_id"].tolist()
    df = df[~df["query_id"].isin(ids_to_drop)]
    print(f"Dropped {len(ids_to_drop)} rows per your resolution")

# Apply any label corrections
for _, row in resolved_A2.iterrows():
    if row.get("drop", False):
        continue
    mask = df["query_id"] == row["query_id"]
    df.loc[mask, "tool_name"] = row["tool_name"]

df.to_csv("human_queries_after_dedup.csv", index=False)
print(f"✅ {len(df)} rows remaining after dedup, saved to human_queries_after_dedup.csv")

In [ ]:
# ── Cell B0: Load post-dedup data, generate the ambiguity review sheet ───────
df = pd.read_csv("human_queries_after_dedup.csv")

to_review = df[(df["ambiguity_level"]=="high") | (df["query_structure"]=="ambiguous_phrasing")].copy()
to_review = to_review.drop_duplicates(subset="query_id")

to_review["acceptable_tool_labels"] = ""
to_review["strict_eval"] = ""
to_review["reviewer_decision_notes"] = ""

to_review = to_review[["query_id","query","tool_name","image_id","objects","ambiguity_level",
                         "query_structure","acceptable_tool_labels","strict_eval","reviewer_decision_notes"]]
to_review.to_csv("review_B_ambiguous_queries.csv", index=False)
print(f"✅ {len(to_review)} rows flagged for manual review → review_B_ambiguous_queries.csv")
print(f"   (high ambiguity: {(df['ambiguity_level']=='high').sum()}, "
      f"ambiguous_phrasing: {(df['query_structure']=='ambiguous_phrasing').sum()}, "
      f"overlap: {((df['ambiguity_level']=='high')&(df['query_structure']=='ambiguous_phrasing')).sum()})")

In [ ]:
# ── Cell B1: Validate your completed review before merging ───────────────────
import pandas as pd 
completed_B = pd.read_csv("/kaggle/input/datasets/deboparna9das/partb-kan1/review_B_ambiguous_queries.csv")

# Sanity checks
missing_strict_eval = completed_B[completed_B["strict_eval"].isna() | (completed_B["strict_eval"]=="")]
print(f"Rows missing strict_eval value: {len(missing_strict_eval)}")
if len(missing_strict_eval):
    print(missing_strict_eval[["query_id","query"]].to_string(index=False))

invalid_strict = completed_B[~completed_B["strict_eval"].astype(str).str.lower().isin(["true","false"])]
print(f"Rows with invalid strict_eval value (must be true/false): {len(invalid_strict)}")

print(f"\nstrict_eval distribution:\n{completed_B['strict_eval'].value_counts()}")
print(f"\nRows with multi-label (acceptable_tool_labels non-empty): {(completed_B['acceptable_tool_labels'].astype(str).str.strip()!='').sum()}")

In [ ]:
# ── Cell B1.5 (safety reload): Ensure df is loaded before merging ────────────
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/deboparna9das/dedup-b/human_queries_after_dedup.csv")
completed_B = pd.read_csv("/kaggle/input/datasets/deboparna9das/partb-kan1/review_B_ambiguous_queries.csv")

print(f"df loaded: {len(df)} rows")
print(f"completed_B loaded: {len(completed_B)} rows")

In [ ]:
# ── Cell B2: Merge Part B decisions back into the main dataframe ─────────────
df["acceptable_tool_labels"] = ""
df["strict_eval"] = True
df["reviewer_decision_notes"] = ""

for _, row in completed_B.iterrows():
    mask = df["query_id"] == row["query_id"]
    df.loc[mask, "acceptable_tool_labels"] = row["acceptable_tool_labels"]
    df.loc[mask, "strict_eval"] = str(row["strict_eval"]).lower() == "true"
    df.loc[mask, "reviewer_decision_notes"] = row["reviewer_decision_notes"]

df.to_csv("human_queries_after_ambiguity_review.csv", index=False)
print(f"✅ Saved. strict_eval=False count: {(~df['strict_eval']).sum()}")
print(f"   Multi-label rows: {(df['acceptable_tool_labels'].astype(str).str.strip()!='').sum()}")

In [ ]:
# ── Cell C0: Generate the full 518-row visual-context review sheet ───────────
df = pd.read_csv("human_queries_after_ambiguity_review.csv")

full_review = df[["query_id","query","tool_name","image_id"]].copy()
full_review["routing_requires_visual_context"] = ""  # you fill: yes / no
full_review.to_csv("review_C_visual_context_ALL_ROWS.csv", index=False)
print(f"✅ {len(full_review)} rows saved → review_C_visual_context_ALL_ROWS.csv")
print("\nReminder — test to apply per row:")
print("Cover the image. Read only the query text. Could you confidently name")
print("the correct TOOL from the sentence alone, with zero knowledge of the photo?")
print("  Yes -> routing_requires_visual_context = no")
print("  No  -> routing_requires_visual_context = yes")

In [ ]:
# ── Cell C1: Validate your completed review ───────────────────────────────────
import pandas as pd
completed_C = pd.read_csv("/kaggle/input/datasets/deboparna9das/part-c/review_C_visual_context_ALL_ROWS.csv")

missing = completed_C[completed_C["routing_requires_visual_context"].isna() |
                        (completed_C["routing_requires_visual_context"].astype(str).str.strip()=="")]
print(f"Rows missing an answer: {len(missing)}")
if len(missing):
    print(missing[["query_id","query"]].to_string(index=False))

invalid = completed_C[~completed_C["routing_requires_visual_context"].astype(str).str.lower().isin(["yes","no"])]
print(f"Rows with invalid value (must be yes/no): {len(invalid)}")
if len(invalid):
    print(invalid[["query_id","routing_requires_visual_context"]].to_string(index=False))

print(f"\nDistribution:\n{completed_C['routing_requires_visual_context'].value_counts()}")

In [ ]:
# ── Safety reload before C2 ───────────────────────────────────────────────────
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/deboparna9das/part-c/human_queries_after_ambiguity_review.csv")
completed_C = pd.read_csv("/kaggle/input/datasets/deboparna9das/part-c/review_C_visual_context_ALL_ROWS.csv")

print(f"df loaded: {len(df)} rows")
print(f"completed_C loaded: {len(completed_C)} rows")

In [ ]:
# ── Cell C2: Merge Part C into the master dataframe, save final cleaned CSV ──
df["routing_requires_visual_context"] = ""
for _, row in completed_C.iterrows():
    mask = df["query_id"] == row["query_id"]
    df.loc[mask, "routing_requires_visual_context"] = str(row["routing_requires_visual_context"]).lower()

df.to_csv("human_queries_cleaned.csv", index=False)
print(f"✅ human_queries_cleaned.csv saved — {len(df)} total rows")
print(f"\nFinal strict_eval distribution:\n{df['strict_eval'].value_counts()}")
print(f"\nFinal routing_requires_visual_context distribution:\n{df['routing_requires_visual_context'].value_counts()}")
print(f"\nMulti-label rows: {(df['acceptable_tool_labels'].astype(str).str.strip()!='').sum()}")

In [ ]:
# ── Cell: Reload everything needed for the summary, from your saved files ────
import pandas as pd
from pathlib import Path

# Point these at wherever your downloaded files actually are in THIS session
# (either re-uploaded as a Kaggle Dataset, or sitting in /kaggle/working if
# you re-uploaded them there directly)
PATHS = {
    "combined": "/kaggle/input/datasets/deboparna9das/csv-processing/human_queries_FINAL_combined.csv",
    "after_dedup": "/kaggle/input/datasets/deboparna9das/dedup-b/human_queries_after_dedup.csv",
    "completed_B": "/kaggle/input/datasets/deboparna9das/partb-kan1/review_B_ambiguous_queries.csv",
    "after_ambiguity": "/kaggle/input/datasets/deboparna9das/part-c/human_queries_after_ambiguity_review.csv",
    "completed_C": "/kaggle/input/datasets/deboparna9das/part-c/review_C_visual_context_ALL_ROWS.csv",
    "final_cleaned": "/kaggle/working/human_queries_cleaned.csv",
}

# Check which ones actually exist before trying to use them
print("File availability check:")
for name, path in PATHS.items():
    exists = Path(path).exists()
    print(f"  {'✅' if exists else '❌'} {name}: {path}")

In [ ]:
# ── Cell: Generate human_queries_cleaned_TRUE_FINAL.csv in this Kaggle session ─
import pandas as pd

# Base file: the A2-resolved version (518 rows, deduped/reworded)
base = pd.read_csv("/kaggle/input/datasets/deboparna9das/parta1/review_A2_label_conflicts_RESOLVED.csv.csv")
# ⚠️ adjust the path above to wherever review_A2_label_conflicts_RESOLVED_csv.csv
#    actually lives in your Kaggle inputs — check with the file-listing cell below
#    if you're not sure of the exact dataset slug

# Merge in Part B's ambiguity decisions
completed_B = pd.read_csv("/kaggle/input/datasets/deboparna9das/partb-kan1/review_B_ambiguous_queries.csv")
base["acceptable_tool_labels"] = ""
base["strict_eval"] = True
base["reviewer_decision_notes"] = ""
for _, row in completed_B.iterrows():
    mask = base["query_id"] == row["query_id"]
    base.loc[mask, "acceptable_tool_labels"] = row["acceptable_tool_labels"]
    base.loc[mask, "strict_eval"] = row["strict_eval"]
    base.loc[mask, "reviewer_decision_notes"] = row["reviewer_decision_notes"]

# Merge in Part C's visual-context decisions
completed_C = pd.read_csv("/kaggle/input/datasets/deboparna9das/part-c/review_C_visual_context_ALL_ROWS.csv")
base["routing_requires_visual_context"] = ""
for _, row in completed_C.iterrows():
    mask = base["query_id"] == row["query_id"]
    base.loc[mask, "routing_requires_visual_context"] = row["routing_requires_visual_context"]

# Validation before saving
assert base["query_id"].duplicated().sum() == 0, "duplicate query_ids!"
assert base["query"].duplicated().sum() == 0, "duplicate query texts remain!"
assert base["routing_requires_visual_context"].isin(["yes","no"]).all(), "invalid visual_context values!"

base.to_csv("/kaggle/working/human_queries_cleaned_TRUE_FINAL.csv", index=False)
print(f"✅ human_queries_cleaned_TRUE_FINAL.csv saved to /kaggle/working/")
print(f"   {len(base)} rows")
print(f"\nstrict_eval: {base['strict_eval'].value_counts().to_dict()}")
print(f"routing_requires_visual_context: {base['routing_requires_visual_context'].value_counts().to_dict()}")
print(f"multi-labeled rows: {(base['acceptable_tool_labels'].astype(str).str.strip()!='').sum()}")

In [ ]:
# ── Cleaner approach: skip the stale intermediate file entirely ──────────────
df_combined = pd.read_csv(PATHS["combined"])
completed_B = pd.read_csv(PATHS["completed_B"])
df_final = pd.read_csv("human_queries_cleaned_TRUE_FINAL.csv")

dupe_groups = df_combined[df_combined.duplicated(subset="query", keep=False)]
conflict_check = dupe_groups.groupby("query")["tool_name"].nunique()
conflicting_queries = conflict_check[conflict_check > 1].index.tolist()

summary = {
    "total_queries_before_cleaning": len(df_combined),
    "exact_duplicates_found": dupe_groups["query"].nunique(),
    "label_conflicts_found": len(conflicting_queries),
    "conflicts_resolved_by_rewording": len(conflicting_queries),  # all 5 resolved this way, confirmed
    "rows_manually_reviewed_ambiguous": len(completed_B),
    "rows_multi_labeled": (completed_B["acceptable_tool_labels"].astype(str).str.strip() != "").sum(),
    "rows_strict_eval_false": (~df_final["strict_eval"]).sum(),
    "final_total_queries": len(df_final),
    "final_duplicate_queries_remaining": df_final["query"].duplicated().sum(),  # should be 0
    "visual_context_yes": (df_final["routing_requires_visual_context"] == "yes").sum(),
    "visual_context_no": (df_final["routing_requires_visual_context"] == "no").sum(),
}
for k, v in summary.items():
    print(f"{k}: {v}")
pd.Series(summary).to_csv("cleaning_summary_stats_FINAL.csv")

In [ ]:
# ── Cell: Merge Part A/B/C into the final CSV for all downstream work ────────
import pandas as pd
from pathlib import Path

# Point these at wherever your 3 files actually are in this session
PATH_A2_RESOLVED = "/kaggle/input/datasets/deboparna9das/parta1/review_A2_label_conflicts_RESOLVED.csv.csv"   # 518 rows, deduped/reworded
PATH_B_COMPLETED = "/kaggle/input/datasets/deboparna9das/partb-kan1/review_B_ambiguous_queries.csv"                # 118 rows, filled
PATH_C_COMPLETED = "/kaggle/input/datasets/deboparna9das/part-c/review_C_visual_context_ALL_ROWS.csv"          # 518 rows, filled

for p in [PATH_A2_RESOLVED, PATH_B_COMPLETED, PATH_C_COMPLETED]:
    print(f"{'✅' if Path(p).exists() else '❌'} {p}")

base = pd.read_csv(PATH_A2_RESOLVED)
completed_B = pd.read_csv(PATH_B_COMPLETED)
completed_C = pd.read_csv(PATH_C_COMPLETED)

base["acceptable_tool_labels"] = ""
base["strict_eval"] = True
base["reviewer_decision_notes"] = ""
for _, row in completed_B.iterrows():
    mask = base["query_id"] == row["query_id"]
    base.loc[mask, "acceptable_tool_labels"] = row["acceptable_tool_labels"]
    base.loc[mask, "strict_eval"] = row["strict_eval"]
    base.loc[mask, "reviewer_decision_notes"] = row["reviewer_decision_notes"]

base["routing_requires_visual_context"] = ""
for _, row in completed_C.iterrows():
    mask = base["query_id"] == row["query_id"]
    base.loc[mask, "routing_requires_visual_context"] = row["routing_requires_visual_context"]

assert base["query_id"].duplicated().sum() == 0, "duplicate query_ids!"
assert base["query"].duplicated().sum() == 0, "duplicate query texts remain!"
assert base["routing_requires_visual_context"].isin(["yes","no"]).all(), "invalid visual_context values!"

base.to_csv("human_queries_cleaned_TRUE_FINAL.csv", index=False)
print(f"\n✅ human_queries_cleaned_TRUE_FINAL.csv — {len(base)} rows — this is your file for everything from here on")

In [ ]:
# ── Cell: Zip everything from the full human-benchmark cleaning pipeline ─────
import zipfile
from datetime import datetime
from pathlib import Path

files_to_zip = [
    # Raw + intermediate
    "/kaggle/input/datasets/deboparna9das/csv-processing/human_queries_FINAL_combined.csv",
    "review_A_exact_duplicates.csv",
    "review_A2_label_conflicts.csv",
    "review_A2_conflicts_CURRENT.csv",
    "/kaggle/input/datasets/deboparna9das/parta1/review_A2_label_conflicts_RESOLVED.csv.csv",

    # Part B
    "/kaggle/input/datasets/deboparna9das/partb-kan1/review_B_ambiguous_queries.csv",

    # Part C
    "/kaggle/input/datasets/deboparna9das/part-c/review_C_visual_context_ALL_ROWS.csv",

    # Final outputs
    "human_queries_cleaned_TRUE_FINAL.csv",
    "cleaning_summary_stats_FINAL.csv",
]

ts = datetime.now().strftime("%Y%m%d_%H%M")
zip_path = Path(f"/kaggle/working/human_benchmark_FINAL_{ts}.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    found, missing = [], []
    for f in files_to_zip:
        p = Path(f)
        if p.exists():
            zf.write(p, p.name)
            found.append(f)
        else:
            missing.append(f)

print(f"✅ {zip_path.name} ({zip_path.stat().st_size/1e3:.1f} KB)")
print(f"\nIncluded ({len(found)}):")
for f in found:
    print(f"  - {f}")
if missing:
    print(f"\n⚠️ Not found in this session, skipped ({len(missing)}):")
    for f in missing:
        print(f"  - {f}")